# Run EXP-01 - BLIP-2 Fusion VQA

Train and evaluate this EXP from the shared mini HDF5 cache copied to local Colab disk.

## 1. Mount Drive and load repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = "/content/blip2-fusion-experiment-vqa"
GITHUB_USER = "<username>"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/{GITHUB_USER}/blip2-fusion-experiment-vqa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git log --oneline -3

## 2. Install dependencies

In [ ]:
!pip install -r requirements.txt -q

import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

import wandb
wandb.login()

## 3. Choose run

In [ ]:
EXP_ID = "01"
RUN_NUMBER = "1"
YOUR_NAME = "ten"
DATA_ROOT = "/content/drive/MyDrive/blip2_project"

CONFIG_FILE = f"configs/exp{EXP_ID}.yaml"
RUN_NAME = f"exp{EXP_ID}_lan{RUN_NUMBER}_{YOUR_NAME}"
CACHE_DIR_NAME = "cache"
DRIVE_CACHE_DIR = f"{DATA_ROOT}/{CACHE_DIR_NAME}"
LOCAL_CACHE_DIR = "/content/blip2_cache"
ANSWER_LIST = f"{DATA_ROOT}/data/ans2idx.json"
OUTPUT_DIR = f"{DATA_ROOT}/checkpoints/{RUN_NAME}"
EVAL_OUTPUT = f"{OUTPUT_DIR}/val_predictions.json"
CHECKPOINT = f"{OUTPUT_DIR}/best_model.pth"

print("Config:", CONFIG_FILE)
print("Run:", RUN_NAME)
print("Drive mini cache:", DRIVE_CACHE_DIR)
print("Local train cache:", LOCAL_CACHE_DIR)
print("Output:", OUTPUT_DIR)

## 4. Copy mini cache to local disk

In [ ]:
import os
import shutil
from pathlib import Path
import h5py

drive_mini = {
    "train": Path(DRIVE_CACHE_DIR) / "train_features_mini.h5",
    "val": Path(DRIVE_CACHE_DIR) / "val_features_mini.h5",
}
local_h5 = {
    "train": Path(LOCAL_CACHE_DIR) / "train_features.h5",
    "val": Path(LOCAL_CACHE_DIR) / "val_features.h5",
}
Path(LOCAL_CACHE_DIR).mkdir(parents=True, exist_ok=True)

def sample_cache(path):
    with h5py.File(path, "r") as f:
        key = next(iter(f.keys()))
        return key, f[key].shape, f[key].dtype

for split, source in drive_mini.items():
    if not source.exists():
        raise FileNotFoundError(f"Missing mini cache artifact: {source}")
    print(split, "Drive sample:", sample_cache(source))
    target = local_h5[split]
    if not target.exists() or target.stat().st_size != source.stat().st_size:
        shutil.copy2(source, target)
    print(split, "Local sample:", sample_cache(target))
print("Question subset sizes come from:", CONFIG_FILE)


## 5. Train

In [ ]:
!python scripts/train.py \
    --config "{CONFIG_FILE}" \
    --run_name "{RUN_NAME}" \
    --data_root "{DATA_ROOT}" \
    --cache_dir "{LOCAL_CACHE_DIR}" \
    --answer_list "{ANSWER_LIST}" \
    --output_dir "{OUTPUT_DIR}"

## 6. Evaluate

In [ ]:
!python scripts/evaluate.py \
    --config "{CONFIG_FILE}" \
    --checkpoint "{CHECKPOINT}" \
    --split val \
    --data_root "{DATA_ROOT}" \
    --cache_dir "{LOCAL_CACHE_DIR}" \
    --answer_list "{ANSWER_LIST}" \
    --output "{EVAL_OUTPUT}"

## 7. Resume

In [ ]:
!python scripts/train.py \
    --config "{CONFIG_FILE}" \
    --run_name "{RUN_NAME}" \
    --data_root "{DATA_ROOT}" \
    --cache_dir "{LOCAL_CACHE_DIR}" \
    --answer_list "{ANSWER_LIST}" \
    --output_dir "{OUTPUT_DIR}" \
    --resume auto